# Lecture 5 — Model Answer
## Distribution Charts: Airbnb London


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

In [ ]:
df = pd.read_csv('../data/airbnb_london.csv')
print(f"Loaded: {len(df)} listings")
print(df.describe().round(1))


In [ ]:
# Cap at 95th percentile — required by exercise rules
p95 = df['price'].quantile(0.95)
df_cap = df.loc[df['price'] <= p95].copy()
print(f"95th percentile price: £{p95:.0f}")
print(df_cap.groupby('room_type')['price'].describe().round(1))


## Task 1 — Histogram: price by room type (overlapping distributions)


In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
df_two = df_cap.loc[df_cap['room_type'].isin(['Entire home/apt', 'Private room'])].copy()

# Medians for reference lines
med_entire  = df_two.loc[df_two['room_type'] == 'Entire home/apt']['price'].median()
med_private = df_two.loc[df_two['room_type'] == 'Private room']['price'].median()

# Colour palette: categorical — two groups, CVD-safe blue/orange
palette = {'Entire home/apt': '#2E75B6', 'Private room': '#E07B39'}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.histogram(
    df_two,
    x='price',
    color='room_type',
    barmode='overlay',
    nbins=60,
    opacity=0.6,                                # overlay opacity set in px
    color_discrete_map=palette,
    labels={'price': 'Nightly Price (£)', 'room_type': 'Room Type'},
    title='Entire homes cost roughly twice as much as private rooms — distributions barely overlap',
)

# ── Step 2: Graph Objects customisation ───────────────────────────────────────
# Median reference line — Entire home
fig.add_vline(
    x=med_entire, line_dash='dash', line_color='#2E75B6', line_width=1.5,
    annotation=dict(
        text=f'Entire median £{med_entire:.0f}',
        font=dict(color='#2E75B6', size=11),
        xanchor='left', yanchor='top', xshift=8,   # 8px buffer to the right of line
    ),
)

# Median reference line — Private room
fig.add_vline(
    x=med_private, line_dash='dash', line_color='#E07B39', line_width=1.5,
    annotation=dict(
        text=f'Private median £{med_private:.0f}',
        font=dict(color='#E07B39', size=11),
        xanchor='right', yanchor='top', xshift=-8,  # 8px buffer to the left of line
    ),
)

# Outlier cap annotation
fig.add_annotation(
    xref='paper', yref='paper', x=0.98, y=0.95,
    text=f'Prices capped at 95th percentile (£{p95:.0f})',
    showarrow=False,
    font=dict(size=10, color='#888888'),
    align='right',
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(title='Room Type', orientation='h', y=1.08),
    margin=dict(l=60, r=40, t=70, b=40),
)
fig.update_xaxes(gridcolor='#EEEEEE')
fig.update_yaxes(gridcolor='#EEEEEE', title='Number of Listings')

fig.show()

## Another Solution for Task 1 - Using KDE

**What is Kernel Density Estimation (KDE)?**

- KDE is a smoothed version of a histogram — it shows the shape of a distribution as a continuous curve rather than discrete bins
- The y-axis shows density, not counts — the area under each curve sums to 1
- KDE is non-parametric — it makes no assumption that the data is normally distributed; it traces whatever shape the data has

In [ ]:
from scipy.stats import gaussian_kde

# ── Data prep ─────────────────────────────────────────────────────────────────
entire  = df_two.loc[df_two['room_type'] == 'Entire home/apt']['price']
private = df_two.loc[df_two['room_type'] == 'Private room']['price']

# Compute KDE curves over a shared x range
x_range = np.linspace(0, df_two['price'].max(), 300)

kde_entire  = gaussian_kde(entire)(x_range)
kde_private = gaussian_kde(private)(x_range)

# Build a long tidy dataframe for px
df_kde = pd.DataFrame({
    'price':     np.tile(x_range, 2),
    'density':   np.concatenate([kde_entire, kde_private]),
    'room_type': ['Entire home/apt'] * 300 + ['Private room'] * 300,
})

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.line(
    data_frame=df_kde,
    x='price',
    y='density',
    color='room_type',
    color_discrete_map=palette,
    labels={'price': 'Nightly Price (£)', 'density': 'Density', 'room_type': 'Room Type'},
    title='Entire homes are right-skewed and expensive — private rooms cluster at lower prices',
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
# Fill under each curve for readability
fig.update_traces(fill='tozeroy', opacity=0.4)

# Median reference lines
fig.add_vline(
    x=med_entire, line_dash='dash', line_color='#2E75B6', line_width=1.5,
    annotation=dict(
        text=f'Entire median £{med_entire:.0f}',
        font=dict(color='#2E75B6', size=11),
        xanchor='left', yanchor='top', xshift=8,
    ),
)

fig.add_vline(
    x=med_private, line_dash='dash', line_color='#E07B39', line_width=1.5,
    annotation=dict(
        text=f'Private median £{med_private:.0f}',
        font=dict(color='#E07B39', size=11),
        xanchor='right', yanchor='top', xshift=-8,
    ),
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(title='Room Type', orientation='h', y=1.08),
    margin=dict(l=60, r=40, t=70, b=40),
)

fig.update_xaxes(gridcolor='#EEEEEE')
fig.update_yaxes(gridcolor='#EEEEEE')

fig.show()

In [ ]:
df_kde

## Task 2 — Box plot: price by neighbourhood


In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
# Sort neighbourhoods by median price — most expensive at top for horizontal plot
neighbourhood_order = df_cap.groupby('neighbourhood')['price'].median().sort_values(ascending=True).index.tolist()


# Identify the two cheapest neighbourhoods (first two in ascending sort)
two_cheapest = neighbourhood_order[:2]
print(f"Two cheapest neighbourhoods: {two_cheapest}")

# Colour role column: highlight the two cheapest
df_cap['highlight'] = df_cap['neighbourhood'].apply(lambda n: 'Cheapest two' if n in two_cheapest else 'Other')

# Colour palette: highlight = orange, rest = grey
palette = {'Cheapest two': '#E07B39', 'Other': '#AAAAAA'}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.box(
    df_cap,
    x='price',
    y='neighbourhood',
    color='highlight',
    category_orders={
        'neighbourhood': neighbourhood_order,   # sorted by median
        'highlight': ['Other', 'Cheapest two'], # legend order
    },
    color_discrete_map=palette,
    labels={'price': 'Nightly Price (£)', 'neighbourhood': ''},
    title=f'{two_cheapest[0]} and {two_cheapest[1]} are the most affordable neighbourhoods in London',
    points='outliers',# show outliers as individual points
    height=700

)

# ── Step 2: Graph Objects customisation ───────────────────────────────────────
fig.update_traces(
    selector=dict(name='Other'),
    marker=dict(opacity=0.4),
)
fig.update_traces(
    selector=dict(name='Cheapest two'),
    marker=dict(opacity=0.8),
)

# Outlier cap annotation
fig.add_annotation(
    xref='paper', yref='paper', x=0.98, y=1.05,
    text=f'Prices capped at 95th percentile (£{p95:.0f})',
    showarrow=False,
    font=dict(size=10, color='#888888'),
    align='right',
)

fig.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    legend=dict(title='', orientation='h', y=1.04),
    margin=dict(l=160, r=40, t=70, b=60),
)
fig.update_xaxes(gridcolor='#EEEEEE')
fig.update_yaxes(gridcolor='#EEEEEE')

fig.show()
